# 2 Node 中访问 State

Node 函数接收 State、返回更新。这一节看 Node 里怎么读 State、怎么写 State。

## 2.1 读取 State

Node 的第一个参数就是 State 本身，按 key 读取。State 是快照：Node 内部就地改它**不会生效**，想改只能靠返回值（见 2.2）。

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class ReadState(TypedDict):
    user: str
    count: int


def greet(state: ReadState) -> dict:
    state["user"] = "aihaipeng"
    state["count"] = 5
    return {}  # 返回值为空则什么状态都不修改


builder = StateGraph(state_schema=ReadState)
builder.add_node("greet", greet)
builder.add_edge(START, "greet")
builder.add_edge("greet", END)

builder.compile().invoke({"user": "Alice", "count": 1})
# 读取 user: Alice
# 读取 count: 1

## 2.2 更新 State

Node 通过**返回值**更新 State：dict 里只写要更新的 key，其余 key 保持不变；带 reducer 的 key 走 reducer 合并。

In [ ]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class UpdateState(TypedDict):
    user: str
    logs: Annotated[list[str], add]


def run(state: UpdateState) -> dict:
    state["user"] = "aihaipeng" # 只修改未输出则不修改 state 中的 user 属性
    return {"logs": ["step 完成"]}  


builder = StateGraph(state_schema=UpdateState)
builder.add_node("run", run)
builder.add_edge(START, "run")
builder.add_edge("run", END)

print(builder.compile().invoke({"user": "Alice", "logs": []}))
# {'user': 'Alice', 'logs': ['step 完成']}

## 2.3 Overwrite 绕过 reducer

带 reducer 的 key 默认走合并；想**整体覆盖**时，把返回值用 `Overwrite(value)` 包一层（清空对话的示例见 1.3.3）。这里用计数器演示：直接返回会累加，`Overwrite` 则整体重置。

In [ ]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Overwrite


class CounterState(TypedDict):
    count: Annotated[int, add]


def inc(state: CounterState) -> dict:
    return {"count": 1}


def reset(state: CounterState) -> dict:
    return {"count": Overwrite(0)}  # 绕过 add，整体重置为 0


builder = StateGraph(state_schema=CounterState)
builder.add_node("inc", inc)
builder.add_node("reset", reset)
builder.add_edge(START, "inc")
builder.add_edge("inc", "reset")
builder.add_edge("reset", END)

print(builder.compile().invoke({"count": 5}))
# {'count': 0}   ← 不带 Overwrite 的话会是 6